In [1]:
import src.dataset_loader as dataset_loader
import cv2
import numpy as np

In [2]:
train_data, test_data = dataset_loader.load_dataset("data/CASIA Iris Image Database (version 1)")


In [3]:
test_iris = test_data[0]

In [4]:
test_iris

('001',
 array([[167, 164, 173, ..., 167, 183, 176],
        [160, 160, 166, ..., 181, 174, 173],
        [163, 164, 163, ..., 172, 172, 178],
        ...,
        [167, 162, 169, ..., 172, 170, 165],
        [159, 159, 166, ..., 171, 182, 190],
        [161, 161, 162, ..., 173, 180, 195]], shape=(280, 320), dtype=uint8))

In [5]:
iris = test_iris[1]
blur = cv2.GaussianBlur(iris, (5, 5), 1.0)
edges = cv2.Canny(blur, 50, 150)

cv2.imshow("gray", iris)
cv2.imshow("blur", blur)
cv2.imshow("edges", edges)
cv2.waitKey(0)
cv2.destroyAllWinows()


AttributeError: module 'cv2' has no attribute 'destroyAllWinows'

In [6]:
def threshold_iris(iris, percentile):
    return np.percentile(iris, percentile)

In [ ]:
threshold = threshold_iris(blur, 5)
dark_region = (blur <= threshold).astype(np.uint8) * 255
#dark_region = cv2.medianBlur(dark_region, 5)

In [10]:
cv2.imshow("iris", iris)
cv2.imshow("blur", blur)
cv2.imshow("dark_region", dark_region)
cv2.waitKey(0)
cv2.destroyAllWindows()



In [ ]:
#coarse center approximation
def rough_pupil_center(image):
    x = np.sum(image, axis=0)
    y = np.sum(image, axis=1)
    x = np.argmin(x)
    y = np.argmin(y)
    return x, y

def find_largest_connected_component(image):
    retval, labels, stats, centroids = cv2.connectedComponentsWithStats(image)
    print(retval)
    if retval <= 1:
        return None 

    largets_label_found = np.argmax(stats[1:, cv2.CC_STAT_AREA]) + 1
    component_found = (labels == largets_label_found).astype(np.uint8) * 255
    return component_found

pupil_x, pupil_y = rough_pupil_center(iris)
#local window
y0 = max(0, pupil_y - 80)
y1 = min(iris.shape[0], pupil_y + 80)
x0 = max(0, pupil_x - 80)
x1 = min(iris.shape[1], pupil_x + 80)

relevant_window = iris[y0:y1, x0:x1]

# adaptive threshold within local window
threshold = np.percentile(relevant_window, 5)
dark_region = (relevant_window <= threshold).astype(np.uint8) * 255
dark_region = cv2.medianBlur(dark_region, 5)

pupil_region = find_largest_connected_component(dark_region)

M = cv2.moments(pupil_region)

centroid_x = int(M["m10"] / M["m00"])
centroid_y = int(M["m01"] / M["m00"])

converted_x = x0 + centroid_x
converted_y = y0 + centroid_y

area = cv2.countNonZero(pupil_region)
radius = int(np.sqrt(area / np.pi))

#cv2.circle(iris, (converted_x, converted_y), 5, (0, 0, 255), -1)


# output = iris.copy()
# cv2.circle(output, (converted_x, converted_y), radius, 255, 2)
# cv2.circle(output, (converted_x, converted_y), 2, 255, -1)

edges = cv2.Canny(iris, 50, 150)

height, width = edges.shape
radii = []
points = []

for theta in np.linspace(0, 2*np.pi, 120, endpoint=False):
    for r in range(int(radius * 2), int(radius * 4)):
        x = int(round(converted_x + r * np.cos(theta)))
        y = int(round(converted_y + r * np.sin(theta)))

        if x < 0 or x >= w or y < 0 or y >= h:
            break

        if edges[y, x] > 0:
            radii.append(r)
            points.append((x, y))
            break

if len(radii) > 0:
    iris_radius = int(np.median(radii))
else:
    iris_radius = None

















2
